# Cost Calculation Examples

This notebook demonstrates how to calculate costs for different Databricks workloads.

## Workload Types

| # | Workload | Compute Type | Formula |
|---|----------|--------------|---------|
| 1 | ETL/Jobs | Classic | DBU Cost + VM Cost |
| 2 | ETL/Jobs | Serverless | DBU Cost (includes compute) |
| 3 | DLT | Core/Pro/Advanced | DBU Cost + VM Cost |
| 4 | DBSQL | Classic/Pro | DBU Cost + VM Cost |
| 5 | DBSQL | Serverless | DBU Cost only |
| 6 | Vector Search | Serverless | Units × DBU Rate × DBU Price |
| 7 | Model Serving | CPU/GPU | DBU Rate × DBU Price |
| 8 | FMAPI | Pay-per-token | Tokens × Rate × DBU Price |
| 9 | FMAPI | Provisioned | Capacity × Rate × DBU Price |


In [ ]:
# Configuration & Tables
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"

TABLES = {
    "instance_rates": f"{CATALOG}.{SCHEMA}.instance_rates",
    "vm_costs": f"{CATALOG}.{SCHEMA}.vm_costs",
    "dbu_prices": f"{CATALOG}.{SCHEMA}.dbu_prices",
    "dbu_multipliers": f"{CATALOG}.{SCHEMA}.dbu_multipliers",
    "dbsql_rates": f"{CATALOG}.{SCHEMA}.dbsql_rates",
    "dbsql_warehouse_config": f"{CATALOG}.{SCHEMA}.dbsql_warehouse_config",
    "serverless_product_rates": f"{CATALOG}.{SCHEMA}.serverless_product_rates",
    "fmapi_databricks_rates": f"{CATALOG}.{SCHEMA}.fmapi_databricks_rates",
    "fmapi_proprietary_rates": f"{CATALOG}.{SCHEMA}.fmapi_proprietary_rates",
}

print("✅ Configuration loaded")


In [ ]:
# =============================================================================
# EXAMPLE PARAMETERS - Change these to calculate different scenarios
# =============================================================================

# Common parameters
CLOUD = "AWS"                    # AWS, AZURE, GCP
REGION = "us-east-1"             # Cloud region code
TIER = "PREMIUM"                 # ENTERPRISE or PREMIUM
HOURS_PER_MONTH = 730

# ETL/Jobs parameters
DRIVER_INSTANCE = "i3.xlarge"
WORKER_INSTANCE = "i3.2xlarge"
NUM_WORKERS = 4
USE_PHOTON = True
JOB_HOURS_PER_DAY = 2
DAYS_PER_MONTH = 30

# DBSQL parameters
WAREHOUSE_SIZE = "Medium"
WAREHOUSE_HOURS_PER_DAY = 8

# Vector Search / Model Serving
VECTOR_SEARCH_TYPE = "standard"
VECTOR_UNITS = 2
MODEL_SERVING_SIZE = "cpu_small"

# FMAPI parameters
FMAPI_MODEL = "llama-3-3-70b"
INPUT_TOKENS_PER_MONTH = 10_000_000
OUTPUT_TOKENS_PER_MONTH = 2_000_000

print(f"✅ Parameters: {CLOUD} / {REGION} / {TIER}")


In [ ]:
# =============================================================================
# 🔍 DEBUG: Check what's available in the tables
# =============================================================================
print("="*70)
print("🔍 AVAILABLE DATA CHECK")
print("="*70)

# Check dbu_multipliers
print("\n📊 dbu_multipliers table:")
try:
    mult_df = spark.sql(f"SELECT * FROM {TABLES['dbu_multipliers']} LIMIT 10")
    if mult_df.count() > 0:
        display(mult_df)
    else:
        print("   ⚠️ Table is EMPTY - Run notebook 09_Load_DBU_Multipliers")
except Exception as e:
    print(f"   ❌ Table doesn't exist: {e}")

# Check dbu_prices (should have all products with region)
print(f"\n📊 JOBS product_types in dbu_prices ({CLOUD}, {REGION}):")
try:
    jobs_types = spark.sql(f"""
        SELECT DISTINCT product_type, pricing_type, price_per_dbu 
        FROM {TABLES['dbu_prices']}
        WHERE UPPER(cloud) = '{CLOUD}' 
        AND region = '{REGION}'
        AND tier = '{TIER}'
        AND product_type LIKE '%JOBS%'
        ORDER BY product_type
    """).toPandas()
    if len(jobs_types) > 0:
        for _, row in jobs_types.iterrows():
            print(f"   {row['product_type']} [{row['pricing_type']}]: ${row['price_per_dbu']}/DBU")
    else:
        print("   ⚠️ No JOBS product types found - Run notebook 01_Fetch_DBU_Prices")
except Exception as e:
    print(f"   ❌ Error: {e}")

# Check instance_rates
print(f"\n📊 Sample instance_rates ({CLOUD}):")
try:
    inst_df = spark.sql(f"""
        SELECT instance_type, dbu_rate FROM {TABLES['instance_rates']}
        WHERE UPPER(cloud) = '{CLOUD}' LIMIT 5
    """).toPandas()
    if len(inst_df) > 0:
        for _, row in inst_df.iterrows():
            print(f"   {row['instance_type']}: {row['dbu_rate']} DBU/hr")
    else:
        print("   ⚠️ No instances found - Run notebook 02_Load_DBU_Rates")
except Exception as e:
    print(f"   ❌ Error: {e}")


In [ ]:
# =============================================================================
# 1️⃣ ETL/JOBS - CLASSIC COMPUTE
# Formula: Total = DBU_Cost + VM_Cost
# DBU_Cost = (driver_dbu + workers × worker_dbu) × photon_mult × dbu_price × hours
# VM_Cost = (driver_vm + workers × worker_vm) × hours
# =============================================================================
print("="*70)
print("1️⃣ ETL/JOBS - CLASSIC COMPUTE")
print("="*70)

def safe_lookup(query, field, description):
    """Safe lookup with error message."""
    result = spark.sql(query).collect()
    if not result:
        print(f"❌ NOT FOUND: {description}")
        print(f"   Query: {query[:200]}...")
        return None
    return result[0][field]

# Get DBU rates
print("\n🔍 Looking up data...")

driver_dbu = safe_lookup(f"""
    SELECT dbu_rate FROM {TABLES['instance_rates']}
    WHERE instance_type = '{DRIVER_INSTANCE}' AND UPPER(cloud) = '{CLOUD}'
""", 'dbu_rate', f"Driver instance {DRIVER_INSTANCE} in instance_rates")

worker_dbu = safe_lookup(f"""
    SELECT dbu_rate FROM {TABLES['instance_rates']}
    WHERE instance_type = '{WORKER_INSTANCE}' AND UPPER(cloud) = '{CLOUD}'
""", 'dbu_rate', f"Worker instance {WORKER_INSTANCE} in instance_rates")

# Get Photon multiplier (note: feature column is lowercase 'photon')
photon_mult = 1.0
if USE_PHOTON:
    pm = safe_lookup(f"""
        SELECT multiplier FROM {TABLES['dbu_multipliers']}
        WHERE UPPER(cloud) = '{CLOUD}' AND LOWER(feature) = 'photon' AND sku_type = 'JOBS_COMPUTE'
    """, 'multiplier', f"Photon multiplier for {CLOUD}")
    photon_mult = pm if pm else 2.0  # Default Photon = 2x if not found
    print(f"   ✓ Photon multiplier: {photon_mult}x")

# Get DBU price from dbu_prices (uses cloud region code directly!)
product_type = "JOBS_COMPUTE_(PHOTON)" if USE_PHOTON else "JOBS_COMPUTE"
dbu_price = safe_lookup(f"""
    SELECT price_per_dbu FROM {TABLES['dbu_prices']}
    WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
    AND tier = '{TIER}' AND product_type = '{product_type}'
""", 'price_per_dbu', f"DBU price for {product_type} in {REGION}")

if dbu_price:
    print(f"   ✓ DBU price: ${dbu_price}/DBU")

# Get VM costs (on_demand)
driver_vm = safe_lookup(f"""
    SELECT cost_per_hour FROM {TABLES['vm_costs']}
    WHERE instance_type = '{DRIVER_INSTANCE}' AND UPPER(cloud) = '{CLOUD}' 
    AND region = '{REGION}' AND pricing_tier = 'on_demand'
""", 'cost_per_hour', f"VM cost for {DRIVER_INSTANCE} in {REGION}")

worker_vm = safe_lookup(f"""
    SELECT cost_per_hour FROM {TABLES['vm_costs']}
    WHERE instance_type = '{WORKER_INSTANCE}' AND UPPER(cloud) = '{CLOUD}' 
    AND region = '{REGION}' AND pricing_tier = 'on_demand'
""", 'cost_per_hour', f"VM cost for {WORKER_INSTANCE} in {REGION}")

# Check if all lookups succeeded
if None in [driver_dbu, worker_dbu, dbu_price, driver_vm, worker_vm]:
    print("\n⚠️ Some lookups failed. Check the tables above.")
    raise SystemExit("Missing data - cannot calculate")

# Calculate
hours = JOB_HOURS_PER_DAY * DAYS_PER_MONTH
total_dbu_hr = (driver_dbu + NUM_WORKERS * worker_dbu) * photon_mult
dbu_cost_hr = total_dbu_hr * dbu_price
vm_cost_hr = driver_vm + NUM_WORKERS * worker_vm

print(f"\n📋 Cluster: 1x {DRIVER_INSTANCE} + {NUM_WORKERS}x {WORKER_INSTANCE}")
print(f"   Photon: {'Yes' if USE_PHOTON else 'No'} ({photon_mult}x)")
print(f"   Total DBU/hr: {total_dbu_hr:.2f}")

print(f"\n💰 HOURLY COST:")
print(f"   DBU:  ${dbu_cost_hr:.4f} ({total_dbu_hr:.2f} DBU × ${dbu_price}/DBU)")
print(f"   VM:   ${vm_cost_hr:.4f} (driver ${driver_vm:.4f} + {NUM_WORKERS}×${worker_vm:.4f})")
print(f"   Total: ${dbu_cost_hr + vm_cost_hr:.4f}")

print(f"\n📊 MONTHLY ({hours} hrs):")
print(f"   DBU Cost:  ${dbu_cost_hr * hours:,.2f}")
print(f"   VM Cost:   ${vm_cost_hr * hours:,.2f}")
print(f"   ─────────────────────")
print(f"   TOTAL:     ${(dbu_cost_hr + vm_cost_hr) * hours:,.2f}")

# Store for comparison
classic_monthly = (dbu_cost_hr + vm_cost_hr) * hours
classic_vm_monthly = vm_cost_hr * hours


In [ ]:
# =============================================================================
# 2️⃣ ETL/JOBS - SERVERLESS (same config, no separate VM cost)
# =============================================================================
print("="*70)
print("2️⃣ ETL/JOBS - SERVERLESS")
print("="*70)

# Serverless DBU price (includes compute)
sl_dbu_price = spark.sql(f"""
    SELECT price_per_dbu FROM {TABLES['dbu_prices']}
    WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
    AND tier = '{TIER}' AND product_type = 'JOBS_SERVERLESS_COMPUTE'
""").collect()[0]['price_per_dbu']

sl_cost_hr = total_dbu_hr * sl_dbu_price
sl_monthly = sl_cost_hr * hours

print(f"\n📋 Same DBU consumption as Classic: {total_dbu_hr:.2f} DBU/hr")
print(f"   Serverless DBU Price: ${sl_dbu_price}/DBU (includes compute)")

print(f"\n💰 HOURLY: ${sl_cost_hr:.4f}")
print(f"📊 MONTHLY: ${sl_monthly:,.2f}")

print(f"\n📈 vs Classic:")
savings = classic_monthly - sl_monthly
pct = (savings / classic_monthly) * 100 if classic_monthly > 0 else 0
print(f"   Classic:    ${classic_monthly:,.2f}")
print(f"   Serverless: ${sl_monthly:,.2f}")
print(f"   Difference: ${savings:+,.2f} ({pct:+.1f}%)")


In [ ]:
# =============================================================================
# 3️⃣ DLT (Delta Live Tables) - Core/Pro/Advanced
# Same instance DBU calculation, different product SKUs
# =============================================================================
print("="*70)
print("3️⃣ DLT - CORE / PRO / ADVANCED")
print("="*70)

print(f"\n📋 Cluster: 1x {DRIVER_INSTANCE} + {NUM_WORKERS}x {WORKER_INSTANCE}")
print(f"   Total DBU/hr: {total_dbu_hr:.2f}, Hours/month: {hours}")
print(f"   VM Cost/month: ${classic_vm_monthly:,.2f}")

print(f"\n{'Edition':<25} {'DBU Price':>10} {'DBU Cost':>12} {'VM Cost':>12} {'TOTAL':>12}")
print(f"{'-'*25} {'-'*10} {'-'*12} {'-'*12} {'-'*12}")

for edition in ["DLT_CORE", "DLT_PRO", "DLT_ADVANCED"]:
    product = f"{edition}_COMPUTE_(PHOTON)" if USE_PHOTON else f"{edition}_COMPUTE"
    try:
        dlt_price = spark.sql(f"""
            SELECT price_per_dbu FROM {TABLES['dbu_prices']}
            WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
            AND tier = '{TIER}' AND product_type = '{product}'
        """).collect()[0]['price_per_dbu']
        
        dlt_dbu_cost = total_dbu_hr * dlt_price * hours
        dlt_total = dlt_dbu_cost + classic_vm_monthly
        
        name = edition.replace("DLT_", "") + (" (Photon)" if USE_PHOTON else "")
        print(f"{name:<25} ${dlt_price:>8.4f} ${dlt_dbu_cost:>10,.2f} ${classic_vm_monthly:>10,.2f} ${dlt_total:>10,.2f}")
    except:
        print(f"{edition:<25} Not available")


In [ ]:
# =============================================================================
# 4️⃣ DBSQL - CLASSIC / PRO (DBU + VM)
# =============================================================================
print("="*70)
print("4️⃣ DBSQL - CLASSIC / PRO")
print("="*70)

dbsql_hours = WAREHOUSE_HOURS_PER_DAY * DAYS_PER_MONTH

for wh_type in ['classic', 'pro']:
    print(f"\n{'─'*35}")
    print(f"   {wh_type.upper()} WAREHOUSE ({WAREHOUSE_SIZE})")
    print(f"{'─'*35}")
    
    # Get DBU rate and SKU type
    info = spark.sql(f"""
        SELECT dbu_per_hour, sku_product_type 
        FROM {TABLES['dbsql_rates']}
        WHERE UPPER(cloud) = '{CLOUD}' AND warehouse_type = '{wh_type}'
        AND warehouse_size = '{WAREHOUSE_SIZE}'
    """).collect()[0]
    
    # Get VM config
    cfg = spark.sql(f"""
        SELECT driver_instance_type, driver_count, worker_instance_type, worker_count
        FROM {TABLES['dbsql_warehouse_config']}
        WHERE UPPER(cloud) = '{CLOUD}' AND warehouse_type = '{wh_type}'
        AND warehouse_size = '{WAREHOUSE_SIZE}'
    """).collect()[0]
    
    # Get DBU price
    dbsql_dbu_price = spark.sql(f"""
        SELECT price_per_dbu FROM {TABLES['dbu_prices']}
        WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
        AND tier = '{TIER}' AND product_type = '{info['sku_product_type']}'
    """).collect()[0]['price_per_dbu']
    
    # Get VM costs
    drv_vm = spark.sql(f"""
        SELECT cost_per_hour FROM {TABLES['vm_costs']}
        WHERE UPPER(instance_type) = UPPER('{cfg['driver_instance_type']}')
        AND UPPER(cloud) = '{CLOUD}' AND region = '{REGION}' AND pricing_tier = 'on_demand'
    """).collect()[0]['cost_per_hour']
    
    wkr_vm = spark.sql(f"""
        SELECT cost_per_hour FROM {TABLES['vm_costs']}
        WHERE UPPER(instance_type) = UPPER('{cfg['worker_instance_type']}')
        AND UPPER(cloud) = '{CLOUD}' AND region = '{REGION}' AND pricing_tier = 'on_demand'
    """).collect()[0]['cost_per_hour']
    
    # Calculate
    dbu_cost_hr = info['dbu_per_hour'] * dbsql_dbu_price
    vm_cost_hr = drv_vm * cfg['driver_count'] + wkr_vm * cfg['worker_count']
    total_hr = dbu_cost_hr + vm_cost_hr
    
    print(f"   DBU/hr: {info['dbu_per_hour']}")
    print(f"   Driver: {cfg['driver_count']}x {cfg['driver_instance_type']} (${drv_vm:.4f}/hr)")
    print(f"   Workers: {cfg['worker_count']}x {cfg['worker_instance_type']} (${wkr_vm:.4f}/hr)")
    
    print(f"\n   💰 HOURLY:  DBU ${dbu_cost_hr:.4f} + VM ${vm_cost_hr:.4f} = ${total_hr:.4f}")
    print(f"   📊 MONTHLY: DBU ${dbu_cost_hr * dbsql_hours:,.2f} + VM ${vm_cost_hr * dbsql_hours:,.2f} = ${total_hr * dbsql_hours:,.2f}")


In [ ]:
# =============================================================================
# 5️⃣ DBSQL - SERVERLESS (DBU only, no VM)
# =============================================================================
print("="*70)
print("5️⃣ DBSQL - SERVERLESS")
print("="*70)

# Get Serverless DBU rate
sl_info = spark.sql(f"""
    SELECT dbu_per_hour, sku_product_type 
    FROM {TABLES['dbsql_rates']}
    WHERE UPPER(cloud) = '{CLOUD}' AND warehouse_type = 'serverless'
    AND warehouse_size = '{WAREHOUSE_SIZE}'
""").collect()[0]

# Get Serverless DBU price
dbsql_sl_price = spark.sql(f"""
    SELECT price_per_dbu FROM {TABLES['dbu_prices']}
    WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
    AND tier = '{TIER}' AND product_type = '{sl_info['sku_product_type']}'
""").collect()[0]['price_per_dbu']

sl_cost_hr = sl_info['dbu_per_hour'] * dbsql_sl_price
sl_monthly = sl_cost_hr * dbsql_hours

print(f"\n📋 {WAREHOUSE_SIZE} Serverless Warehouse")
print(f"   DBU/hr: {sl_info['dbu_per_hour']}")
print(f"   DBU Price: ${dbsql_sl_price}/DBU (includes compute)")

print(f"\n💰 HOURLY:  ${sl_cost_hr:.4f}")
print(f"📊 MONTHLY: ${sl_monthly:,.2f} ({dbsql_hours} hrs)")


In [ ]:
# =============================================================================
# 6️⃣ VECTOR SEARCH
# Formula: units × dbu_rate × dbu_price × hours
# =============================================================================
print("="*70)
print("6️⃣ VECTOR SEARCH")
print("="*70)

# Get Vector Search rate
vs_info = spark.sql(f"""
    SELECT dbu_rate, sku_product_type, description
    FROM {TABLES['serverless_product_rates']}
    WHERE UPPER(cloud) = '{CLOUD}' AND product = 'vector_search'
    AND size_or_model = '{VECTOR_SEARCH_TYPE}'
""").collect()[0]

# Get DBU price
vs_dbu_price = spark.sql(f"""
    SELECT price_per_dbu FROM {TABLES['dbu_prices']}
    WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
    AND tier = '{TIER}' AND product_type = '{vs_info['sku_product_type']}'
""").collect()[0]['price_per_dbu']

vs_cost_hr = VECTOR_UNITS * vs_info['dbu_rate'] * vs_dbu_price
vs_monthly = vs_cost_hr * HOURS_PER_MONTH

print(f"\n📋 Vector Search ({VECTOR_SEARCH_TYPE})")
print(f"   Units: {VECTOR_UNITS}")
print(f"   DBU/unit/hr: {vs_info['dbu_rate']}")
print(f"   Total DBU/hr: {VECTOR_UNITS * vs_info['dbu_rate']}")
print(f"   DBU Price: ${vs_dbu_price}/DBU")

print(f"\n💰 HOURLY:  ${vs_cost_hr:.4f}")
print(f"📊 MONTHLY: ${vs_monthly:,.2f} (24/7 = {HOURS_PER_MONTH} hrs)")


In [ ]:
# =============================================================================
# 7️⃣ MODEL SERVING (CPU/GPU)
# =============================================================================
print("="*70)
print("7️⃣ MODEL SERVING (CPU/GPU)")
print("="*70)

serving_hours = HOURS_PER_MONTH  # 24/7

# Get all serving options (product = 'model_serving', size_or_model has cpu/gpu variants)
serving_df = spark.sql(f"""
    SELECT size_or_model, dbu_rate, sku_product_type, description, product
    FROM {TABLES['serverless_product_rates']}
    WHERE UPPER(cloud) = '{CLOUD}'
    AND product = 'model_serving'
    ORDER BY dbu_rate
""").toPandas()

print(f"\n📋 Model Serving Options ({CLOUD}):")
print(f"   {'Size':<30} {'DBU/hr':>10} {'$/hr':>12} {'$/month':>12}")
print(f"   {'-'*30} {'-'*10} {'-'*12} {'-'*12}")

if len(serving_df) == 0:
    print("   ⚠️ No model serving options found in serverless_product_rates")
    print(f"      Check if table has product='cpu_model_serving' or 'gpu_model_serving' for {CLOUD}")
else:
    for _, row in serving_df.iterrows():
        try:
            ms_price = spark.sql(f"""
                SELECT price_per_dbu FROM {TABLES['dbu_prices']}
                WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
                AND tier = '{TIER}' AND product_type = '{row['sku_product_type']}'
            """).collect()[0]['price_per_dbu']
            
            cost_hr = row['dbu_rate'] * ms_price
            cost_mo = cost_hr * serving_hours
            print(f"   {row['size_or_model']:<30} {row['dbu_rate']:>10.2f} ${cost_hr:>10.4f} ${cost_mo:>10,.2f}")
        except Exception as e:
            print(f"   {row['size_or_model']:<30} {row['dbu_rate']:>10.2f} (price N/A: {row['sku_product_type']})")


In [ ]:
# =============================================================================
# 8️⃣ FMAPI - PAY PER TOKEN (Databricks Models)
# Formula: (input_tokens × input_rate + output_tokens × output_rate) × dbu_price / 1M
# =============================================================================
print("="*70)
print("8️⃣ FMAPI - PAY PER TOKEN")
print("="*70)

# Get token rates (rate_type = 'input_token' and 'output_token' are separate rows)
token_rates = spark.sql(f"""
    SELECT 
        model,
        MAX(CASE WHEN rate_type = 'input_token' THEN dbu_rate END) as input_dbu_per_1m,
        MAX(CASE WHEN rate_type = 'output_token' THEN dbu_rate END) as output_dbu_per_1m,
        MAX(sku_product_type) as sku_product_type
    FROM {TABLES['fmapi_databricks_rates']}
    WHERE UPPER(cloud) = '{CLOUD}' AND model = '{FMAPI_MODEL}' 
    AND rate_type IN ('input_token', 'output_token')
    GROUP BY model
""").collect()

if token_rates:
    tr = token_rates[0]
    
    # Get DBU price
    fmapi_price = spark.sql(f"""
        SELECT price_per_dbu FROM {TABLES['dbu_prices']}
        WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
        AND tier = '{TIER}' AND product_type = '{tr['sku_product_type']}'
    """).collect()[0]['price_per_dbu']
    
    # Calculate
    input_dbu = (INPUT_TOKENS_PER_MONTH / 1_000_000) * tr['input_dbu_per_1m']
    output_dbu = (OUTPUT_TOKENS_PER_MONTH / 1_000_000) * tr['output_dbu_per_1m']
    total_dbu = input_dbu + output_dbu
    total_cost = total_dbu * fmapi_price
    
    print(f"\n📋 Model: {FMAPI_MODEL}")
    print(f"   Input rate:  {tr['input_dbu_per_1m']} DBU/1M tokens")
    print(f"   Output rate: {tr['output_dbu_per_1m']} DBU/1M tokens")
    print(f"   DBU Price:   ${fmapi_price}/DBU")
    
    print(f"\n📊 Usage:")
    print(f"   Input:  {INPUT_TOKENS_PER_MONTH:,} tokens = {input_dbu:,.2f} DBU")
    print(f"   Output: {OUTPUT_TOKENS_PER_MONTH:,} tokens = {output_dbu:,.2f} DBU")
    
    print(f"\n💰 MONTHLY COST:")
    print(f"   Input:  ${input_dbu * fmapi_price:,.2f}")
    print(f"   Output: ${output_dbu * fmapi_price:,.2f}")
    print(f"   ─────────────────────")
    print(f"   TOTAL:  ${total_cost:,.2f}")
else:
    print(f"❌ Model '{FMAPI_MODEL}' not found in fmapi_databricks_rates")


In [ ]:
# =============================================================================
# 9️⃣ FMAPI - PROVISIONED THROUGHPUT
# Formula: capacity_units × dbu_rate × dbu_price × hours
# =============================================================================
print("="*70)
print("9️⃣ FMAPI - PROVISIONED THROUGHPUT")
print("="*70)

PROVISIONED_UNITS = 2

# Get provisioned rates (rate_type = 'provisioned_entry' or 'provisioned_scaling')
# dbu_rate = DBU per capacity unit per hour
prov_rates = spark.sql(f"""
    SELECT model, 
           MAX(CASE WHEN rate_type = 'provisioned_entry' THEN dbu_rate END) as entry_rate,
           MAX(CASE WHEN rate_type = 'provisioned_scaling' THEN dbu_rate END) as scaling_rate,
           MAX(sku_product_type) as sku_product_type
    FROM {TABLES['fmapi_databricks_rates']}
    WHERE UPPER(cloud) = '{CLOUD}' AND model = '{FMAPI_MODEL}' AND rate_type LIKE 'provisioned%'
    GROUP BY model
""").collect()

if prov_rates:
    pr = prov_rates[0]
    
    # Get DBU price
    prov_price = spark.sql(f"""
        SELECT price_per_dbu FROM {TABLES['dbu_prices']}
        WHERE UPPER(cloud) = '{CLOUD}' AND region = '{REGION}'
        AND tier = '{TIER}' AND product_type = '{pr['sku_product_type']}'
    """).collect()[0]['price_per_dbu']
    
    # Calculate (1 entry + remaining as scaling)
    entry_units = min(1, PROVISIONED_UNITS)
    scaling_units = max(0, PROVISIONED_UNITS - 1)
    entry_rate = pr['entry_rate'] or pr['scaling_rate']
    
    total_dbu_hr = entry_units * entry_rate + scaling_units * pr['scaling_rate']
    cost_hr = total_dbu_hr * prov_price
    cost_monthly = cost_hr * HOURS_PER_MONTH
    
    print(f"\n📋 Model: {FMAPI_MODEL}")
    print(f"   Entry rate:   {entry_rate} DBU/unit/hr (first unit)")
    print(f"   Scaling rate: {pr['scaling_rate']} DBU/unit/hr (additional)")
    print(f"   DBU Price:    ${prov_price}/DBU")
    
    print(f"\n📊 Capacity: {PROVISIONED_UNITS} units")
    print(f"   Entry:   {entry_units} × {entry_rate} = {entry_units * entry_rate} DBU/hr")
    print(f"   Scaling: {scaling_units} × {pr['scaling_rate']} = {scaling_units * pr['scaling_rate']} DBU/hr")
    print(f"   Total:   {total_dbu_hr} DBU/hr")
    
    print(f"\n💰 COST (24/7):")
    print(f"   Hourly:  ${cost_hr:.4f}")
    print(f"   Monthly: ${cost_monthly:,.2f}")
else:
    print(f"❌ Provisioned throughput not available for '{FMAPI_MODEL}'")


In [ ]:
# =============================================================================
# 📊 SUMMARY - All Tables Used
# =============================================================================
print("="*70)
print("📊 TABLES USED FOR COST CALCULATION")
print("="*70)

print("""
┌─────────────────────────────┬────────────────────────────────────────────┐
│ Table                       │ Purpose                                    │
├─────────────────────────────┼────────────────────────────────────────────┤
│ dbu_prices                  │ DBU price per SKU/region/tier              │
│ sku_region_mapping          │ Map SKU region → cloud region code         │
│ instance_rates              │ DBU/hr rate per instance type              │
│ vm_costs                    │ VM cost per hour (on-demand/spot/reserved) │
│ dbu_multipliers             │ Photon/Graviton multipliers                │
│ dbsql_rates                 │ DBSQL DBU/hr by warehouse size             │
│ dbsql_warehouse_config      │ DBSQL driver/worker VM config              │
│ serverless_product_rates    │ Vector Search, Model Serving DBU rates     │
│ fmapi_databricks_rates      │ FMAPI token & provisioned rates            │
│ fmapi_proprietary_rates     │ OpenAI/Anthropic/Google rates              │
└─────────────────────────────┴────────────────────────────────────────────┘

KEY JOINS:
  • instance_rates.instance_type → vm_costs.instance_type (+ cloud + region)
  • dbu_prices.region → sku_region_mapping.sku_region (+ cloud)
  • *_rates.sku_product_type → dbu_prices.product_type (+ cloud + region + tier)
""")
